In [ ]:
'''
5000 Resumes
	Data Science
	Data Engineering
	Software Engineering
	CAD Resumes
	Devops Resumes

Task 1 : either a ml model or gen ai model 
	inputs : 5000 Resumes
	output : category 

task2 : chromdb collection for these 5000 documents and create persist 
task3 : create the metadata and filter based on the metadata
task4: job description==>predict which category =>filtering in chroma db==> top 3 matched resumes
'''

In [4]:
# ==========================================
# STEP 1: Dependencies & Environment Setup
# ==========================================
%pip install pypdf
%pip install sentence_transformers
import joblib
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Install required PDF extraction library if not present
# %pip install pypdf

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 12.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
# ==========================================
# STEP 2: Extract Text from Job Descriptions
# ==========================================

reader = PdfReader("../../resources/JD/20_Job_Descriptions.pdf")
jd_data = []

# Iterate through each page of the multi-role JD PDF
for page_num, page_data in enumerate(reader.pages, start=1):
    text_lines = page_data.extract_text().split("\n")
    
    if page_num == 1:
        # First page contains the main header; strip it out
        jd_text = ' '.join(text_lines[1:])
        jd_role = text_lines[1]
    else:
        # Subsequent pages standard structure
        jd_text = ' '.join(text_lines[:])
        jd_role = text_lines[0]
        
    jd_data.append({
        "jd_page": page_num, 
        "jd_role": jd_role, 
        "jd_text": jd_text
    })

# Preview the second extracted JD role
jd_data[1]

{'jd_page': 2,
 'jd_role': 'Azure DevOps Engineer',
 'jd_text': "Azure DevOps Engineer Job Summary We are seeking a skilled Azure DevOps Engineer to join our team and contribute to designing, developing, and supporting business-critical solutions. Key Responsibilities \x7f Design, develop, and maintain enterprise solutions. \x7f Collaborate with cross-functional teams and stakeholders. \x7f Troubleshoot issues and optimize performance. \x7f Follow best practices for quality, security, and compliance. \x7f Prepare technical documentation and reports. Required Qualifications Bachelor's degree in a relevant field and 2+ years of professional experience. Preferred Skills Problem-solving, communication, teamwork, cloud technologies, automation, and data-driven decision making. "}

In [7]:
# ==========================================
# STEP 3: Batch Load & Process Resumes
# ==========================================

import glob
import os

print("### READING AVAILABLE RESUMES")
resume_data = []

# Batch process all raw PDF resumes in target directory
for file in glob.glob("../../resources/resumes/*.pdf"):
    text = ""
    reader = PdfReader(file)
    
    # Extract structural text page by page
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            # Append trailing space to prevent boundary word merging
            text += page_text + " "
            
    resume_data.append({
        "resume_name": os.path.basename(file), 
        "resume_text": text
    })

print(f"Loaded {len(resume_data)} resumes successfully.\n")

### READING AVAILABLE RESUMES
Loaded 5000 resumes successfully.



In [8]:
# ==========================================
# STEP 4: Initialize Embedding Model & Encode
# ==========================================

# Initialize a highly performant embedding model (BGE family)
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# 4a. Embed Job Descriptions
jd_texts = [jd["jd_text"] for jd in jd_data]
jd_embeddings = model.encode(jd_texts, normalize_embeddings=True, show_progress_bar=True)
print(f"JD Embeddings Shape: {jd_embeddings.shape}")

# 4b. Embed Resumes (Batched processing for memory optimization)
resume_texts = [resume["resume_text"] for resume in resume_data]
resume_embeddings = model.encode(
    resume_texts, 
    batch_size=32, 
    normalize_embeddings=True, 
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

JD Embeddings Shape: (20, 384)


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [10]:
# ==========================================
# STEP 5: Local Storage & ChromaDB Indexing
# ==========================================
%pip install chromadb
import chromadb
import numpy as np

# Save generated embeddings to disk to prevent re-computation overhead
os.makedirs("./embeddings", exist_ok=True)
np.save("./embeddings/resume_embeddings_bge.npy", resume_embeddings)

# Initialize Persistent Chroma Client
client = chromadb.PersistentClient(path="./chroma_resume_db")
resume_collection = client.get_or_create_collection(name="resume_collection")

# Prepare payloads for Chroma schema
resume_ids = [f"resume_{idx}" for idx in range(len(resume_data))]
resume_documents = [resume["resume_text"] for resume in resume_data]
resume_metadatas = [{"file_name": resume["resume_name"]} for resume in resume_data]

# Batch-upload vectors and text blocks into ChromaDB
batch_size = 100
for i in range(0, len(resume_ids), batch_size):
    resume_collection.add(
        ids=resume_ids[i : i + batch_size],
        documents=resume_documents[i : i + batch_size],
        embeddings=resume_embeddings[i : i + batch_size].tolist(),
        metadatas=resume_metadatas[i : i + batch_size]
    )
    print(f"Indexed up to document ID: {min(i + batch_size, len(resume_ids))}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 5.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 3.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 5.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 4.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 4.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.1 MB/s eta 0:00:00
   ━━━━

In [11]:
# ==========================================
# STEP 6: Execute Vector Search Queries
# ==========================================

# Target Semantic Search Query
query = "Azure Data Engineer Databricks ADF Spark Data Lake"

# Generate normalized embedding vectors for input query
query_embedding = model.encode(query, normalize_embeddings=True)

# Query ChromaDB Vector Index for Top 5 relevant semantic matches
results = resume_collection.query(
    query_embeddings=[query_embedding.tolist()], 
    n_results=5
)

In [12]:
user_query = input("Enter your search: ")
user_query_embedding = model.encode(user_query, normalize_embeddings=True)

# Query ChromaDB Vector Index for Top 5 relevant semantic matches
user_results = resume_collection.query(
    query_embeddings=[user_query_embedding.tolist()], 
    n_results=5
)

user_results

Enter your search:  Azure data engineering


{'ids': [['resume_64',
   'resume_1314',
   'resume_4965',
   'resume_3427',
   'resume_4256']],
 'embeddings': None,
 'documents': [["Erin Harmon\nerin.harmon34175@pena-daniels.info | 9819463955 | PSC 4014, Box 5149, APO AE 35198 |\nlinkedin.com/in/erin-harmon-34175\nPROFESSIONAL SUMMARY\nExperienced professional with a strong background in Education and expertise in Azure, DevOps, Machine\nLearning. Proven track record of delivering high-quality results in fast-paced environments. Excellent\nTeamwork, Critical Thinking skills with a passion for continuous learning and professional development.\nPROFESSIONAL EXPERIENCE\nData Analyst | Schmidt-Mayo | 2021 - 2024\n\x7f Government development medical go such name woman interest whatever everyone present.\n\x7f Society drug pattern start trade student long worker answer particular go.\n\x7f Value lawyer team wish.\n\x7f Require analysis it find risk lot under.\n\x7f Hair spend him discussion attack.\n\x7f Next thank degree executive shake

In [ ]:
# 📘 Project Notes: AI-Powered Resume Matching using ChromaDB

## 🎯 Project Objective

Build a semantic resume search system that can match resumes against job descriptions using embeddings and a vector database.

### Input

- 20 Job Descriptions (JDs)
- 5000 Resumes

### Output

For a given JD:

```text
Azure Data Engineer
        ↓
Top Matching Resumes
```

Example:

```text
1. Resume_221.pdf
2. Resume_875.pdf
3. Resume_1022.pdf
```

---

# Why Not Traditional Keyword Search?

Traditional search:

```text
JD:
Azure Data Engineer

Resume:
Worked with Databricks, ADF, Spark
```

Problem:

```text
Azure Data Engineer
≠
Databricks
```

Keyword matching may fail.

---

Semantic Search:

```text
Azure Data Engineer
≈
Databricks
≈
Spark
≈
Data Lake
≈
ADF
```

Meaning is captured using embeddings.

---

# High Level Architecture

```text
                Job Descriptions
                        ↓
                 Generate Embeddings
                        ↓

5000 Resumes → Generate Embeddings → ChromaDB

                        ↑
                        ↓

               Similarity Search
                        ↓

              Top Matching Resumes
```

---

# Step 1: Extract Job Descriptions

Source:

```python
20_Job_Descriptions.pdf
```

Each page represents one job description.

Example:

```python
{
    "jd_page": 1,
    "jd_role": "Azure Data Engineer",
    "jd_text": "..."
}
```

Output:

```python
jd_data
```

---

# Step 2: Extract Resume Text

Read all resumes:

```python
../../resources/resumes/*.pdf
```

Extract text page-by-page.

Output:

```python
resume_data
```

Structure:

```python
{
    "resume_name": "resume_001.pdf",
    "resume_text": "..."
}
```

---

# Step 3: Generate Embeddings

Model Used:

```python
BAAI/bge-small-en-v1.5
```

Purpose:

Convert text into vectors.

Example:

```text
Azure Data Engineer
```

becomes

```python
[0.12, -0.55, 0.91, ...]
```

A 384-dimensional vector.

---

# Why Embeddings?

Embeddings capture semantic meaning.

Example:

```text
Spark
Kafka
ETL
```

will be closer to:

```text
Data Engineering
```

than:

```text
Digital Marketing
```

even if exact words differ.

---

# Embedding Generation

```python
resume_embeddings = model.encode(
    resume_texts,
    normalize_embeddings=True
)
```

Output:

```python
(5000, 384)
```

Meaning:

```text
5000 resumes
384 dimensions per resume
```

---

# Why Normalize Embeddings?

```python
normalize_embeddings=True
```

Benefits:

- Unit length vectors
- Better similarity calculations
- More stable retrieval

---

# Step 4: Save Embeddings

Embeddings are expensive to generate.

Store:

```python
np.save(...)
```

File:

```text
resume_embeddings.npy
```

Benefits:

```text
Notebook Restart
↓
Load Embeddings
↓
No Reprocessing
```

---

# Step 5: Create ChromaDB

Vector Database:

```python
chromadb.PersistentClient(...)
```

Purpose:

Store embeddings and perform fast similarity search.

---

# Why ChromaDB?

Without Chroma:

```text
JD
↓
Compare against 5000 resumes manually
```

Slow.

---

With Chroma:

```text
JD
↓
Embedding
↓
Vector Search
↓
Top Matches
```

Fast.

---

# Persistent Client

```python
client = chromadb.PersistentClient(
    path="../../resources/chroma_db"
)
```

Benefits:

```text
Data survives notebook restart
```

---

# Step 6: Create Collection

Collection:

```python
resume_collection
```

Think of it as:

```text
SQL Table
```

but for vectors.

---

# Data Stored

For every resume:

### ID

```python
resume_123
```

### Document

```python
Full Resume Text
```

### Embedding

```python
[0.12, -0.55, ...]
```

### Metadata

```python
{
    "file_name":"resume_123.pdf"
}
```

---

# Step 7: Load Data into Chroma

Inserted in batches:

```python
batch_size = 100
```

Reason:

```text
5000 resumes
```

are too large for a single insert.

---

# Step 8: Semantic Search

Example Query:

```text
Azure Data Engineer
Databricks
ADF
Spark
Data Lake
```

Generate embedding:

```python
query_embedding
```

Search:

```python
resume_collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)
```

---

# What Happens Internally?

```text
Query
↓
Embedding
↓
Compare against 5000 Resume Vectors
↓
Find Closest Vectors
↓
Return Top Matches
```

---

# Chroma Query Result

Returns:

```python
{
    "ids": ...,
    "documents": ...,
    "metadatas": ...,
    "distances": ...
}
```

---

# Understanding Distances

Smaller distance:

```text
Better Match
```

Example:

```text
0.15
```

Excellent match.

---

Larger distance:

```text
Poor Match
```

Example:

```text
1.80
```

Weak match.

---

# Important Observation About JDs

JD Similarity Matrix showed:

```text
Azure Data Engineer
≈
Azure DevOps Engineer
≈
Cloud Engineer
```

Very high similarity.

Reason:

```text
Most JDs share the same template
```

Examples:

```text
Collaborate with teams
Problem solving
Communication
Documentation
```

appear in many JDs.

---

# Current Design Decision

We are NOT classifying resumes.

We are using:

```text
Resume Retrieval
```

instead of:

```text
Resume Classification
```

---

# Why?

A resume may fit multiple roles.

Example:

```text
Spark
Databricks
ETL
Azure
```

can match:

- Azure Data Engineer
- Big Data Engineer
- ETL Developer

simultaneously.

---

# Retrieval Architecture

```text
Resume
↓
Embedding
↓
Store in Chroma
```

When JD arrives:

```text
JD
↓
Embedding
↓
Search Chroma
↓
Top Matching Resumes
```

---

# Relation to RAG

Current Project:

```text
JD
↓
Embedding
↓
Vector Search
↓
Relevant Resumes
```

RAG:

```text
Question
↓
Embedding
↓
Vector Search
↓
Relevant Documents
↓
LLM
↓
Answer
```

Retrieval step is identical.

---

# Future Enhancements

## Store JDs in Separate Collection

```text
resume_collection
jd_collection
```

---

## Add Metadata

Example:

```python
{
    "experience": 5,
    "location": "Delhi",
    "skills": "Python"
}
```

---

## Metadata Filtering

```python
where={
    "experience": {
        "$gte": 5
    }
}
```

---

## Gemini Re-ranking

Current:

```text
JD
↓
Chroma
↓
Top 20 Resumes
```

Enhanced:

```text
JD
↓
Chroma
↓
Top 20 Resumes
↓
Gemini
↓
Top 3 Resumes
```

---

# Key Learnings

✅ PDF Processing

✅ Resume Extraction

✅ Sentence Transformer Embeddings

✅ Vector Representations

✅ Embedding Persistence

✅ ChromaDB

✅ Vector Collections

✅ Metadata

✅ Semantic Search

✅ Similarity Retrieval

✅ Resume Matching

✅ Foundations of RAG

---

# Interview Questions

### What is an embedding?

A dense numerical representation of text that captures semantic meaning.

---

### Why use embeddings instead of keywords?

Embeddings capture meaning, not just exact words.

---

### What is a vector database?

A database optimized for storing and searching embeddings.

---

### Why use ChromaDB?

To efficiently store embeddings and perform semantic similarity search.

---

### What is the role of metadata?

Provides additional information for filtering and retrieval.

---

### Why save embeddings?

To avoid recomputing expensive embeddings every time.

---

### How does Chroma find similar resumes?

By comparing vector distances between the query embedding and stored resume embeddings.

---

### How is this related to RAG?

Both systems use embeddings and vector search to retrieve relevant information before further processing.